# HW2 · Part 1 — Data Cleaning & Validation

**Goal:** turn the deliberately dirty `ohlc_hw` table into a trustworthy `ohlc_hw_cleaned` table, with every repair flagged and every removed row accounted for.

**How to run:** from `Homeworks/HW2/Theo_Basseras/`, with the course's database copied to `data/data.db`, run this notebook top to bottom (or `papermill 01_data_cleaning.ipynb 01_data_cleaning.ipynb`). It never modifies `ohlc_hw` or `ohlc`; it (re)writes `ohlc_hw_cleaned`, `ohlc_hw_quarantine` and `ohlc_hw_ticker_audit`.

**What the diagnosis found** (all numbers are recomputed below):

| Problem | Rows | Handling |
|---|---|---|
| Ticker labels wrapped in HTML markup (`<a ...><span ...>KO</span></a>`) | ~40k | Labels normalized; an audit table keeps every original label |
| Exact duplicate records | 29,460 | Removed (after normalization) |
| Conflicting `(ticker, ts)` keys | 0 | Would be quarantined, never "keep first" |
| Missing `adj_close` | ~18k | Rebuilt from the validated raw close × the reference adjustment factor |
| Corrupted `adj_close` (4–24× too large) | ~10k | Detected without the reference, then rebuilt the same way |
| `open = 0` on a ticker's first observation | 5 | Quarantined (no earlier price to repair from) |
| Extreme returns (3σ) | ~5k | Kept and flagged (the raw prices agree with the supplied `ohlc` table) |
| One-day bad prints (prices ×2, zero volume, reversed next day) | 3 | Interpolated between neighbours, flagged |
| Zero-volume days | 75 | Kept and flagged (possible stale data) |
| Missing cells in raw OHLC/volume, missing trading days, impossible OHLC ranges, non-finite prices, zero, negative or infinite adjusted closes, conflicting keys | 0 | Detectors and repairs still run; their behaviour is shown on small injected examples |

After every repair step, the OHLC bounds and the adjusted-close consistency are rechecked (table in 1.3).

In [1]:
# Parameters
db_path = "data/data.db"
raw_table = "ohlc_hw"          # dirty homework table (never modified)
reference_table = "ohlc"       # supplied table, used only for adjustment factors and as a calendar
cleaned_table = "ohlc_hw_cleaned"
quarantine_table = "ohlc_hw_quarantine"
audit_table = "ohlc_hw_ticker_audit"

In [2]:
import html
import re
import sqlite3
import warnings

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

PRICE_COLS = ["open", "high", "low", "close"]
ALL_PRICE_COLS = PRICE_COLS + ["adj_close"]
VALUE_COLS = ALL_PRICE_COLS + ["volume"]
KEY = ["ticker", "ts"]

## 1.1 Load and inspect the data

Columns: `ts` = observation date (ISO string), `ticker`, raw `open`/`high`/`low`/`close` (split-adjusted vendor prices, not dividend-adjusted), vendor `adj_close` (split- and dividend-adjusted close) and `volume` (shares traded).

Some ticker labels are wrapped in HTML markup, so the list of tickers below uses `normalize_ticker` (markup, HTML entities, whitespace and case removed); the markup itself is reported in 1.2.

In [3]:
with sqlite3.connect(db_path) as con:
    raw = pd.read_sql_query(f"SELECT * FROM {raw_table}", con)
    reference = pd.read_sql_query(f"SELECT * FROM {reference_table}", con)

raw_row_count = len(raw)
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


In [4]:
raw.describe()

,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


In [5]:
def normalize_ticker(label: str) -> str:
    """Return the plain ticker symbol: strip HTML markup and entities, whitespace and case."""
    text = html.unescape(str(label))
    text = re.sub(r"<[^>]+>", "", text)
    return re.sub(r"\s+", "", text).upper()


iso_dates = raw["ts"].astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}") & pd.to_datetime(
    raw["ts"], format="%Y-%m-%d", errors="coerce").notna()
print(f"Date range: {raw['ts'].min()} to {raw['ts'].max()} "
      f"({int((~iso_dates).sum())} dates not in ISO YYYY-MM-DD format)")
tickers = raw["ticker"].map(normalize_ticker)
print(f"{raw['ticker'].nunique()} distinct ticker labels = {tickers.nunique()} tickers once the markup is removed:")
print(", ".join(sorted(tickers.unique())))
raw.assign(ticker=tickers).groupby("ticker")["ts"].agg(first_date="min", last_date="max", raw_rows="size")

Date range: 1962-01-02 to 2024-12-06 (0 dates not in ISO YYYY-MM-DD format)


150 distinct ticker labels = 30 tickers once the markup is removed:
AAPL, AMGN, AMZN, AXP, BA, CAT, CRM, CSCO, CVX, DIS, GS, HD, HON, IBM, JNJ, JPM, KO, MCD, MMM, MRK, MSFT, NKE, NVDA, PG, SHW, TRV, UNH, V, VZ, WMT


,first_date,last_date,raw_rows
ticker,,,
AAPL,1980-12-12,2024-10-18,11942
AMGN,1983-06-17,2024-10-18,11290
AMZN,1997-05-15,2024-12-06,7458
AXP,1972-04-03,2024-10-18,14260
BA,1962-01-02,2024-10-18,16980
CAT,1962-01-02,2024-10-18,17044
CRM,2004-06-23,2024-10-18,5525
CSCO,1990-02-16,2024-10-18,9427
CVX,1962-01-02,2024-12-06,17073


## 1.2 Detect data quality issues

Everything in this section is **diagnosis only**: no row is changed. Each detector returns issue rows in a common format, which together form the Data Quality Report (`ticker | issue_type | issue_date | description | severity`). One observation can carry several issues.

**Severity scale**
- **high**: the value cannot be used as is (impossible or nonpositive price, missing or corrupted adjusted close used by the reports, conflicting duplicate keys).
- **medium**: the dataset would be distorted if left untreated, but every value is intact (exact duplicates, markup in ticker labels, missing trading days).
- **low**: worth a look, but not an error by itself (3σ return outliers, zero-volume days).

In [6]:
def issue_rows(frame: pd.DataFrame, issue_type: str, description, severity: str) -> pd.DataFrame:
    """Build report rows (one per row of `frame`) in the common report format."""
    if frame.empty:
        descriptions = []
    elif callable(description):
        descriptions = frame.apply(description, axis=1)
    else:
        descriptions = [description] * len(frame)
    return pd.DataFrame({
        "ticker": frame["ticker"].to_numpy(),
        "issue_type": issue_type,
        "issue_date": frame["ts"].to_numpy(),
        "description": np.asarray(descriptions, dtype=object),
        "severity": severity,
    })

In [7]:
def detect_ticker_markup(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Find labels that differ from their normalized ticker. Returns (audit table, issue rows per label)."""
    audit = (df.assign(normalized=df["ticker"].map(normalize_ticker))
               .groupby(["ticker", "normalized"]).size().rename("rows").reset_index()
               .rename(columns={"ticker": "original_label", "normalized": "ticker"}))
    dirty = audit[audit["original_label"] != audit["ticker"]]
    issues = pd.DataFrame({
        "ticker": dirty["ticker"].to_numpy(),
        "issue_type": "ticker_markup",
        "issue_date": None,
        "description": [f"label {lab!r} used on {n} rows" for lab, n in zip(dirty["original_label"], dirty["rows"])],
        "severity": "medium",
    })
    return audit, issues


def detect_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Exact duplicate records (medium) and conflicting values for the same (ticker, ts) key (high)."""
    exact = df[df.duplicated(KEY + VALUE_COLS, keep="first")]
    unique_rows = df.drop_duplicates(KEY + VALUE_COLS)
    conflicting = unique_rows[unique_rows.duplicated(KEY, keep=False)]
    return pd.concat([
        issue_rows(exact, "duplicate_exact", "exact copy of another record", "medium"),
        issue_rows(conflicting, "duplicate_conflict", "same (ticker, ts) with different values", "high"),
    ], ignore_index=True)


def detect_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """One issue row per missing cell in a price or volume column."""
    out = []
    for col in VALUE_COLS:
        rows = df[df[col].isna()]
        severity = "high" if col in ALL_PRICE_COLS else "medium"
        out.append(issue_rows(rows, f"missing_{col}", f"{col} is missing", severity))
    return pd.concat(out, ignore_index=True)

In [8]:
def detect_missing_trading_days(df: pd.DataFrame, calendar: pd.DataFrame) -> pd.DataFrame:
    """Dates present in a ticker's reference calendar, within its own history, but absent from `df`.

    Assumption: the supplied `ohlc` dates for each ticker are the trading calendar over that ticker's
    history (first to last observation). Business weekdays alone would wrongly include exchange holidays.
    """
    out = []
    for ticker, cal in calendar.groupby("ticker"):
        have = df.loc[df["ticker"] == ticker, "ts"]
        if have.empty:
            continue
        expected = cal.loc[(cal["ts"] >= have.min()) & (cal["ts"] <= have.max()), "ts"]
        missing = sorted(set(expected) - set(have))
        if missing:
            out.append(pd.DataFrame({
                "ticker": ticker, "issue_type": "missing_trading_day", "issue_date": missing,
                "description": "no row for a date in the reference calendar", "severity": "medium",
            }))
    if not out:
        return pd.DataFrame(columns=["ticker", "issue_type", "issue_date", "description", "severity"])
    return pd.concat(out, ignore_index=True)


def missing_day_summary(issues: pd.DataFrame, df: pd.DataFrame) -> list[str]:
    """Sentences such as 'Ticker X is missing Y trading days between [start] and [end]'."""
    lines = []
    for ticker, span in df.groupby("ticker")["ts"].agg(["min", "max"]).iterrows():
        n = int((issues["ticker"] == ticker).sum()) if len(issues) else 0
        lines.append(f"Ticker {ticker} is missing {n} trading days between {span['min']} and {span['max']}")
    return lines

In [9]:
def daily_simple_returns(df: pd.DataFrame, price_col: str = "close") -> pd.Series:
    """(close - prev_close) / prev_close within each ticker, on rows sorted by ticker and date."""
    ordered = df.sort_values(KEY)
    return ordered.groupby("ticker")[price_col].pct_change(fill_method=None).reindex(df.index)


def detect_return_outliers(df: pd.DataFrame, k: float = 3.0) -> pd.DataFrame:
    """Flag |r - mean| > k * std within each ticker (sample std, ddof=1), both tails.

    Undefined returns (first row of a ticker, missing or nonpositive previous price) are excluded.
    A ticker whose std is zero or undefined cannot be screened: it gets one 'return_std_undefined' row
    instead of flags. This is a retrospective review flag, not a deletion rule.
    """
    r = daily_simple_returns(df).replace([np.inf, -np.inf], np.nan)
    stats = r.groupby(df["ticker"]).agg(["mean", lambda x: x.std(ddof=1)])
    stats.columns = ["mean", "std"]
    mean = df["ticker"].map(stats["mean"])
    std = df["ticker"].map(stats["std"])
    usable = r.notna() & std.notna() & (std > 0)
    flagged = usable & ((r - mean).abs() > k * std)
    rows = df[flagged].assign(_r=r[flagged], _z=((r - mean) / std)[flagged])
    report = issue_rows(rows, "return_outlier",
                        lambda x: f"return {x['_r']:+.2%} is {x['_z']:+.1f} std from the ticker mean", "low")
    unscreened = stats.index[~(stats["std"] > 0)]   # std of 0 or NaN
    if len(unscreened) == 0:
        return report
    notes = pd.DataFrame({
        "ticker": unscreened, "issue_type": "return_std_undefined", "issue_date": None,
        "description": "return std is zero or undefined, so the 3-sigma screen cannot run", "severity": "low",
    })
    return pd.concat([report, notes], ignore_index=True) if len(report) else notes


def detect_price_problems(df: pd.DataFrame) -> pd.DataFrame:
    """Impossible OHLC relationships and nonpositive / non-finite prices (adj_close included)."""
    out = []
    checks = {
        "high_below_close": df["high"] < df["close"],
        "low_above_close": df["low"] > df["close"],
        "open_outside_range": (df["open"] < df["low"]) | (df["open"] > df["high"]),
        "high_below_low": df["high"] < df["low"],
    }
    for name, mask in checks.items():
        out.append(issue_rows(df[mask], name, f"impossible price relationship ({name})", "high"))
    for col in ALL_PRICE_COLS:
        values = df[col]
        bad = values.notna() & (~np.isfinite(values) | (values <= 0))
        out.append(issue_rows(df[bad].assign(_v=values[bad]), f"invalid_{col}",
                              lambda x, c=col: f"{c} = {x['_v']} (must be finite and positive)", "high"))
    return pd.concat(out, ignore_index=True)


def detect_zero_volume(df: pd.DataFrame) -> pd.DataFrame:
    """Zero-volume days: not impossible, but a sign of stale or placeholder data."""
    return issue_rows(df[df["volume"] == 0], "zero_volume", "volume is 0 (possible stale data)", "low")

In [10]:
def adjustment_factor_spikes(df: pd.DataFrame, window: int = 7, ratio: float = 1.5) -> pd.Series:
    """Boolean mask of suspicious adj_close values, found without any reference data.

    The adjustment factor adj_close / close is a slowly moving step function at most 1 (dividends and
    splits only ever lower past adjusted prices). A value is suspicious when its factor exceeds 1, or
    exceeds `ratio` times the median factor of its neighbours (`window` rows centred on it, same ticker).
    Zero, negative or infinite adjusted closes are invalid rather than spikes (see `invalid_adjusted_close`);
    they are left out here, so they can't distort the neighbours' median either.
    """
    ordered = df.sort_values(KEY)
    factor = ordered["adj_close"] / ordered["close"]
    factor = factor.where(np.isfinite(factor) & (factor > 0))
    half = window // 2

    def neighbour_median(s: pd.Series) -> pd.Series:
        """Median of the neighbours in a centred window, the row itself excluded, missing values ignored."""
        padded = np.concatenate([np.full(half, np.nan), s.to_numpy(dtype=float), np.full(half, np.nan)])
        windows = sliding_window_view(padded, window).copy()
        windows[:, half] = np.nan
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)   # windows with no valid neighbour give NaN
            return pd.Series(np.nanmedian(windows, axis=1), index=s.index)

    med = factor.groupby(ordered["ticker"], group_keys=False).apply(neighbour_median)
    mask = factor.notna() & ((factor > 1 + 1e-6) | (factor > ratio * med))
    return mask.reindex(df.index, fill_value=False)


def detect_adj_close_spikes(df: pd.DataFrame) -> pd.DataFrame:
    """Report rows whose adjusted close looks corrupted (see `adjustment_factor_spikes`)."""
    mask = adjustment_factor_spikes(df)
    rows = df[mask].assign(_f=(df["adj_close"] / df["close"])[mask])
    return issue_rows(rows, "adj_close_spike",
                      lambda x: f"adj_close / close = {x['_f']:.2f} (a factor above 1 is impossible)", "high")


def invalid_adjusted_close(df: pd.DataFrame) -> pd.Series:
    """Adjusted closes that are present but zero, negative or infinite (missing ones are counted separately)."""
    adj = df["adj_close"]
    return adj.notna() & (~np.isfinite(adj) | (adj <= 0))

### Run the diagnosis on the raw table

Tickers are normalized *before* grouping, deduplicating or computing returns, otherwise one stock would be split into several pseudo-tickers. The diagnosis below runs on a normalized copy; the raw table stays untouched.

In [11]:
audit, markup_issues = detect_ticker_markup(raw)
work = raw.assign(ticker=raw["ticker"].map(normalize_ticker))
print(f"{raw['ticker'].nunique()} labels -> {work['ticker'].nunique()} tickers")
audit.sort_values("rows", ascending=False).head(8)

150 labels -> 30 tickers


,original_label,ticker,rows
143,PG,PG,15450
139,MRK,MRK,15442
133,IBM,IBM,15433
129,DIS,DIS,15420
134,JNJ,JNJ,15399
136,KO,KO,15385
125,CAT,CAT,15352
128,CVX,CVX,15349


In [12]:
dedup_view = work.drop_duplicates(KEY + VALUE_COLS)   # duplicates would distort the other detectors
calendar = reference[["ticker", "ts"]]

report_before = pd.concat([
    markup_issues,
    detect_duplicates(work),
    detect_missing_values(dedup_view),
    detect_missing_trading_days(dedup_view, calendar),
    detect_return_outliers(dedup_view),
    detect_price_problems(dedup_view),
    detect_zero_volume(dedup_view),
    detect_adj_close_spikes(dedup_view),
], ignore_index=True)

summary_before = (report_before.groupby(["issue_type", "severity"]).size()
                  .rename("issues").reset_index().sort_values("issues", ascending=False))
summary_before

,issue_type,severity,issues
1,duplicate_exact,medium,29460
3,missing_adj_close,high,18374
0,adj_close_spike,high,10418
5,return_outlier,low,5056
6,ticker_markup,medium,120
7,zero_volume,low,75
2,invalid_open,high,5
4,open_outside_range,high,5


In [13]:
print("Missing values in each column:")
pd.DataFrame({"raw table": raw[["ts", "ticker"] + VALUE_COLS].isna().sum(),
              "after removing exact duplicates": dedup_view[["ts", "ticker"] + VALUE_COLS].isna().sum()})

Missing values in each column:


,raw table,after removing exact duplicates
ts,0,0
ticker,0,0
open,0,0
high,0,0
low,0,0
close,0,0
adj_close,19855,18374
volume,0,0


In [14]:
missing_days = report_before[report_before["issue_type"] == "missing_trading_day"]
print("\n".join(missing_day_summary(missing_days, dedup_view)))
print(f"\nMissing trading days in total: {len(missing_days)} (calendar: the supplied ohlc dates of each ticker)")
report_before.sample(8, random_state=0)   # a few rows of the Data Quality Report

Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticke

,ticker,issue_type,issue_date,description,severity
14530,MRK,duplicate_exact,1965-04-27,exact copy of another record,medium
12584,CAT,duplicate_exact,1988-10-13,exact copy of another record,medium
7891,CVX,duplicate_exact,1986-06-17,exact copy of another record,medium
38971,HON,missing_adj_close,2014-09-11,adj_close is missing,high
11077,TRV,duplicate_exact,1996-11-08,exact copy of another record,medium
11025,JPM,duplicate_exact,1995-03-23,exact copy of another record,medium
9048,CSCO,duplicate_exact,2006-06-28,exact copy of another record,medium
56044,V,adj_close_spike,2023-05-25,adj_close / close = 5.97 (a factor above 1 is ...,high


### Detectors on small injected examples

Several required cases do not occur in this database: missing raw prices and volume, missing trading days, high < close, low > close, non-finite or negative prices, conflicting `(ticker, ts)` keys, and a ticker whose returns have no spread. A detector that finds nothing is only convincing if it demonstrably finds the fault when it exists, so each one is run on tiny hand-made tables.

In [15]:
demo = pd.DataFrame({
    "ticker": ["DEMO"] * 7,
    "ts": ["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-08", "2024-01-09", "2024-01-10"],
    "open": [10.0, 10.2, np.nan, 10.4, 10.5, 10.6, 10.2],
    "high": [10.5, 10.6, np.nan, 10.3, 10.9, 11.0, 10.4],     # 2024-01-05: high 10.3 < close 10.5
    "low": [9.8, 10.0, np.nan, 10.1, 10.2, 10.4, 10.1],
    "close": [10.1, 10.4, np.nan, 10.5, 10.7, 10.8, np.nan],  # 01-04: every raw price missing; 01-10: close only
    "adj_close": [10.0, 10.3, np.nan, 10.4, 10.6, 10.7, 10.2],
    "volume": [1000, 1200, 900, np.nan, 1100, 1000, 950],     # 2024-01-05: volume missing
})
demo_calendar = pd.DataFrame({"ticker": "DEMO", "ts": demo["ts"]})
demo_missing_day = demo[demo["ts"] != "2024-01-08"]           # one date deliberately removed
demo_keys = pd.DataFrame({                                   # 03-01 copied twice; 03-04 has two different closes
    "ticker": "DEMO", "ts": ["2024-03-01", "2024-03-01", "2024-03-04", "2024-03-04"],
    "open": [20.0, 20.0, 20.5, 20.5], "high": [20.8, 20.8, 21.0, 21.0], "low": [19.9, 19.9, 20.2, 20.2],
    "close": [20.4, 20.4, 20.9, 20.6], "adj_close": [20.2, 20.2, 20.7, 20.4], "volume": [500, 500, 600, 600],
})
demo_prices = pd.DataFrame({                                 # 04-02: infinite high; 04-03: low > close, adj < 0
    "ticker": "DEMO", "ts": ["2024-04-01", "2024-04-02", "2024-04-03"],
    "open": [30.0, 30.2, 30.1], "high": [30.5, np.inf, 30.6], "low": [29.8, 29.9, 30.4],
    "close": [30.3, 30.1, 30.2], "adj_close": [30.0, 29.8, -1.0], "volume": [700, 800, 900],
})
demo_flat = pd.DataFrame({"ticker": "FLAT", "ts": ["2024-05-01", "2024-05-02", "2024-05-03"],
                          "close": [10.0, 10.0, 10.0]})       # constant price: the return std is 0
pd.concat([
    detect_missing_values(demo),
    detect_price_problems(demo),
    detect_missing_trading_days(demo_missing_day, demo_calendar),
    detect_duplicates(demo_keys),
    detect_price_problems(demo_prices),
    detect_return_outliers(demo_flat),
], ignore_index=True)

,ticker,issue_type,issue_date,description,severity
0,DEMO,missing_open,2024-01-04,open is missing,high
1,DEMO,missing_high,2024-01-04,high is missing,high
2,DEMO,missing_low,2024-01-04,low is missing,high
3,DEMO,missing_close,2024-01-04,close is missing,high
4,DEMO,missing_close,2024-01-10,close is missing,high
5,DEMO,missing_adj_close,2024-01-04,adj_close is missing,high
6,DEMO,missing_volume,2024-01-05,volume is missing,medium
7,DEMO,high_below_close,2024-01-05,impossible price relationship (high_below_close),high
8,DEMO,open_outside_range,2024-01-05,impossible price relationship (open_outside_ra...,high
9,DEMO,missing_trading_day,2024-01-08,no row for a date in the reference calendar,medium


## 1.3 Clean the data

Order matters: normalize labels → remove exact duplicates → quarantine conflicting keys → drop rows with *all* raw prices missing → repair prices within each ticker (sorted by date) → rebuild adjusted close (also for rows whose raw prices were repaired) → interpolate confirmed bad prints → flag what stays → re-validate. The OHLC bounds and the adjusted-close consistency are rechecked after every repair step. Every repair sets a flag column that is saved with the data; rows that cannot be repaired defensibly go to a quarantine table with a reason.

In [16]:
def remove_exact_duplicates(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Drop exact duplicate records (after ticker normalization). Returns (frame, rows removed)."""
    out = df.drop_duplicates(KEY + VALUE_COLS)
    return out, len(df) - len(out)


def split_conflicting_keys(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Move every row of a (ticker, ts) key that has conflicting values to quarantine (no 'keep first')."""
    conflict = df.duplicated(KEY, keep=False)
    return df[~conflict], df[conflict].assign(reason="conflicting values for the same (ticker, ts)")


def drop_rows_without_prices(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Rows whose four raw prices are all missing carry no information: remove them before any filling."""
    empty = df[PRICE_COLS].isna().all(axis=1)
    return df[~empty], df[empty].assign(reason="all raw prices missing")


def bounds_broken(df: pd.DataFrame) -> pd.Series:
    """Rows whose raw prices break low <= open, close <= high (missing values are not counted here)."""
    top, bottom = df[["open", "close"]].max(axis=1), df[["open", "close"]].min(axis=1)
    return (df["high"] < top) | (df["low"] > bottom) | (df["high"] < df["low"])


def mark_impossible_prices(df: pd.DataFrame) -> pd.DataFrame:
    """Blank rows whose prices break the OHLC bounds or are nonpositive or non-finite (Option 1: NaN, then forward-fill).

    Which value is wrong cannot be known from one row, so the whole raw OHLC set of the row is blanked
    and later carried forward from the last valid day of the same ticker.
    """
    out = df.copy()
    values = out[PRICE_COLS]
    invalid = (values.notna() & (~np.isfinite(values) | (values <= 0))).any(axis=1)
    bad = bounds_broken(out) | invalid
    out["flag_ohlc_repaired"] = bad
    out.loc[bad, PRICE_COLS] = np.nan
    return out


def fill_prices_within_ticker(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Forward-fill missing raw prices from the previous valid day of the same ticker, sorted by date.

    Filling one column can break today's range (yesterday's close may lie above today's high), so the
    bounds are rechecked after the fill, and a row that breaks them takes the previous day's whole bar.
    Leading gaps (no earlier valid price) are not extrapolated: those rows are quarantined.
    """
    out = df.sort_values(KEY).copy()
    missing = out[PRICE_COLS].isna().any(axis=1)
    out["flag_price_filled"] = missing
    out[PRICE_COLS] = out.groupby("ticker")[PRICE_COLS].ffill()
    broken = missing & bounds_broken(out)
    out.loc[broken, PRICE_COLS] = np.nan
    out[PRICE_COLS] = out.groupby("ticker")[PRICE_COLS].ffill()
    leading = out[PRICE_COLS].isna().any(axis=1)
    return out[~leading], out[leading].assign(reason="invalid or missing price before the ticker's first valid day")


def fill_missing_volume(df: pd.DataFrame) -> pd.DataFrame:
    """Unknown volume is set to 0 so the column stays numeric, with a flag: it is not evidence of no trading."""
    out = df.copy()
    out["flag_volume_filled"] = out["volume"].isna()
    out["volume"] = out["volume"].fillna(0)
    return out

In [17]:
def rebuild_adjusted_close(df: pd.DataFrame, reference: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Recompute adj_close as raw close x the date's reference adjustment factor wherever it cannot be trusted.

    Source: the supplied `ohlc` table, matched on (ticker, ts); a factor adj_close / close belongs to its
    date only, so no factor is carried across dates. Two cases are recomputed:
    - adj_close missing, invalid (zero, negative or infinite) or corrupted (a factor spike), only when the raw
      close equals the reference close;
    - raw prices repaired by the previous steps (Option 1 or forward fill): the vendor adj_close belonged to
      the original prices, so it is recomputed from the repaired close to keep one adjustment convention.
    A suspected spike whose value already equals the reference is a false alarm and is kept unchanged.
    Rows without a usable reference factor are quarantined.
    """
    out = df.copy()
    ref = reference[KEY + ["close", "adj_close"]].rename(columns={"close": "ref_close", "adj_close": "ref_adj"})
    out = out.merge(ref, on=KEY, how="left", validate="one_to_one")
    factor = out["ref_adj"] / out["ref_close"]
    validated = out["ref_close"].notna() & np.isclose(out["close"], out["ref_close"], rtol=1e-9, atol=1e-12)
    repaired = pd.Series(False, index=out.index)
    for col in ("flag_ohlc_repaired", "flag_price_filled"):
        if col in out:
            repaired |= out[col].astype(bool)
    usable = np.isfinite(factor) & (factor > 0) & (validated | repaired)

    missing = out["adj_close"].isna()
    invalid = invalid_adjusted_close(out)
    spike = adjustment_factor_spikes(out)
    false_alarm = spike & np.isclose(out["adj_close"], out["ref_adj"], rtol=1e-9, atol=1e-12)
    corrupted = spike & ~false_alarm
    to_rebuild = missing | invalid | corrupted | repaired

    out["flag_adj_filled"] = missing & usable
    out["flag_adj_invalid_repaired"] = invalid & usable
    out["flag_adj_spike_repaired"] = corrupted & usable
    out["flag_adj_recomputed"] = repaired & usable
    out.loc[to_rebuild & usable, "adj_close"] = out["close"] * factor

    unresolved = to_rebuild & ~usable
    quarantined = out[unresolved].assign(reason="adj_close missing, invalid, corrupted or stale, and no usable reference factor")
    kept = out[~unresolved].drop(columns=["ref_close", "ref_adj"])
    print(f"adj_close rebuilt: {int(out['flag_adj_filled'].sum())} missing, "
          f"{int(out['flag_adj_invalid_repaired'].sum())} zero, negative or infinite, "
          f"{int(out['flag_adj_spike_repaired'].sum())} corrupted, "
          f"{int(out['flag_adj_recomputed'].sum())} after a raw-price repair; "
          f"{int(false_alarm.sum())} suspected spike(s) matched the reference and were kept; "
          f"{int(unresolved.sum())} unresolved")
    return kept, quarantined.drop(columns=["ref_close", "ref_adj"])


def find_bad_prints(df: pd.DataFrame, jump: float = 0.25, revert: float = 0.05) -> pd.Series:
    """Confirmed one-day data errors: a close that jumps by more than `jump` and is fully reversed the next day.

    Evidence required on top of the jump: the next close is within `revert` of the previous close, and
    the day has zero volume (no trading supports the price). Only interior days qualify.
    """
    ordered = df.sort_values(KEY)
    g = ordered.groupby("ticker")["close"]
    prev_close, next_close = g.shift(1), g.shift(-1)
    jumped = (ordered["close"] / prev_close - 1).abs() > jump
    reverted = (next_close / prev_close - 1).abs() < revert
    mask = jumped & reverted & ordered["volume"].eq(0)
    return mask.reindex(df.index, fill_value=False)


def interpolate_confirmed_errors(df: pd.DataFrame, error_mask: pd.Series, cols=("close",)) -> pd.DataFrame:
    """Linear interpolation of confirmed data errors, inside a sorted ticker, between valid neighbours only.

    Leading or trailing errors are left as NaN (no extrapolation) for the caller to quarantine.
    """
    out = df.sort_values(KEY).copy()
    mask = error_mask.reindex(out.index, fill_value=False)
    for col in cols:
        out.loc[mask, col] = np.nan
        out[col] = out.groupby("ticker")[col].transform(lambda s: s.interpolate(limit_area="inside"))
    out["flag_interpolated"] = mask
    return out.reindex(df.index)


def unresolved_adjustment(df: pd.DataFrame, reference: pd.DataFrame) -> pd.Series:
    """Rows whose adjusted close is still invalid after the repairs, or still looks corrupted and differs from the reference."""
    ref_adj = df[KEY].merge(reference[KEY + ["adj_close"]], on=KEY, how="left")["adj_close"].to_numpy()
    matches = pd.Series(np.isclose(df["adj_close"].to_numpy(), ref_adj, rtol=1e-9, atol=1e-12), index=df.index)
    return invalid_adjusted_close(df) | (adjustment_factor_spikes(df) & ~matches)


def consistency_counts(df: pd.DataFrame) -> dict[str, int]:
    """Recheck run after every repair step; every count except `rows` must be 0 once the cleaning is done."""
    values = df[PRICE_COLS]
    return {
        "rows": len(df),
        "missing raw price": int(values.isna().any(axis=1).sum()),
        "nonpositive or non-finite raw price": int((values.notna() & (~np.isfinite(values) | (values <= 0))).any(axis=1).sum()),
        "OHLC bounds broken": int(bounds_broken(df).sum()),
        "missing adj_close": int(df["adj_close"].isna().sum()),
        "zero, negative or infinite adj_close": int(invalid_adjusted_close(df).sum()),
        "adj_close factor above 1 or spike": int(adjustment_factor_spikes(df).sum()),
    }


def with_original_values(quarantined: pd.DataFrame, original: pd.DataFrame) -> pd.DataFrame:
    """Quarantined rows with the values they had before any repair (a repair step may have blanked them)."""
    return quarantined.drop(columns=VALUE_COLS).merge(original[KEY + VALUE_COLS], on=KEY, how="left")

### Extreme returns: investigation and decision

The 3σ screen flags about 5,000 daily moves, so each needs a reason before anything is changed.

1. **Cross-check:** the raw prices of every flagged row are identical to the supplied `ohlc` table. The homework copy did not alter them, but the two tables come from the same vendor, so this does not prove the prices are right.
2. **Pattern check:** a genuine move persists; a bad print jumps and comes straight back. Three days fit the bad-print pattern exactly (TRV on 1976-10-28, 1978-12-20 and 1981-08-10): all four prices double, the volume is 0, and the next day returns to the previous level. These are **confirmed data errors** and are interpolated between their neighbours (prices and adjusted close), with `flag_interpolated`.
3. **Everything else is kept and flagged** (`flag_return_outlier`). Example: MCD falls 75% on 1968-01-02 and stays at the new level (and so does the adjusted close). That could be an unadjusted split in the vendor history, but nothing in the data proves it, so it is kept, flagged and left as a stated uncertainty. No split or news event is claimed without evidence.

In [18]:
flagged = report_before[report_before["issue_type"] == "return_outlier"][["ticker", "issue_date"]]
flagged = flagged.rename(columns={"issue_date": "ts"})
check = (flagged.merge(dedup_view, on=KEY)
         .merge(reference[KEY + PRICE_COLS], on=KEY, suffixes=("", "_ref")))
same = np.all([np.isclose(check[c], check[f"{c}_ref"], rtol=1e-9) for c in PRICE_COLS], axis=0)
print(f"{len(check)} flagged returns; raw prices identical to the reference on {int(same.sum())} of them")
print("Largest absolute moves among them:")
top = report_before[report_before["issue_type"] == "return_outlier"].copy()
top["abs_ret"] = top["description"].str.extract(r"return ([+-][\d.]+)%").astype(float).abs()
top.nlargest(5, "abs_ret")[["ticker", "issue_date", "description"]]

5056 flagged returns; raw prices identical to the reference on 5056 of them
Largest absolute moves among them:


,ticker,issue_date,description
49883,TRV,1981-08-10,return +100.00% is +41.2 std from the ticker mean
52834,TRV,1976-10-28,return +100.00% is +41.2 std from the ticker mean
48097,TRV,1978-12-20,return +99.75% is +41.1 std from the ticker mean
52772,MCD,1968-01-02,return -75.36% is -40.7 std from the ticker mean
49437,AAPL,2000-09-29,return -51.87% is -18.7 std from the ticker mean


In [19]:
bad_prints = find_bad_prints(dedup_view)
context = dedup_view.sort_values(KEY)
for idx in dedup_view.index[bad_prints]:
    row = dedup_view.loc[idx]
    window = context[(context["ticker"] == row["ticker"])]
    pos = window.index.get_loc(idx)
    print(window.iloc[pos - 1: pos + 2][KEY + PRICE_COLS + ["adj_close", "volume"]].to_string(), "\n")

       ticker          ts  open  high   low  close  adj_close  volume
379629    TRV  1978-12-19  4.02  4.02  4.02   4.02    0.90645   72000
8975      TRV  1978-12-20  8.03  8.03  8.03   8.03    2.80000       0
273555    TRV  1978-12-21  4.09  4.09  4.09   4.09    0.92409  105600 

       ticker          ts   open   high    low  close  adj_close  volume
353858    TRV  1981-08-07   6.03   6.03   6.03   6.03       1.57  264800
146175    TRV  1981-08-10  12.06  12.06  12.06  12.06       4.20       0
148374    TRV  1981-08-11   6.03   6.03   6.03   6.03       1.57   24800 

       ticker          ts  open  high   low  close  adj_close  volume
156463    TRV  1976-10-27  4.61  4.61  4.61   4.61   13.88325  230400
384157    TRV  1976-10-28  9.22  9.22  9.22   9.22        NaN       0
273517    TRV  1976-10-29  4.69  4.69  4.69   4.69    0.99707   79200 



### Run the cleaning pipeline

In [20]:
quarantine = []
checks = {}
step_counts = {"raw rows": len(raw)}

clean = raw.assign(ticker=raw["ticker"].map(normalize_ticker))
clean, n_dupes = remove_exact_duplicates(clean)
step_counts["exact duplicates removed"] = n_dupes

clean, q = split_conflicting_keys(clean); quarantine.append(q)
step_counts["conflicting-key rows quarantined"] = len(q)
clean, q = drop_rows_without_prices(clean); quarantine.append(q)
step_counts["all-prices-missing rows dropped"] = len(q)
checks["before repairs"] = consistency_counts(clean)

before_repair = clean.copy()   # original values, to keep them in the quarantine record
clean = mark_impossible_prices(clean)
clean, q = fill_prices_within_ticker(clean); quarantine.append(with_original_values(q, before_repair))
clean = fill_missing_volume(clean)
checks["after price repairs (Option 1 + forward fill)"] = consistency_counts(clean)

clean, q = rebuild_adjusted_close(clean, reference); quarantine.append(with_original_values(q, before_repair))
checks["after adj_close rebuild"] = consistency_counts(clean)

errors = find_bad_prints(clean)
clean = interpolate_confirmed_errors(clean, errors, cols=PRICE_COLS + ["adj_close"])
step_counts["confirmed bad prints interpolated"] = int(clean["flag_interpolated"].sum())
edge = clean[ALL_PRICE_COLS].isna().any(axis=1)        # a confirmed error with no valid neighbour on one side
stale = unresolved_adjustment(clean, reference) & ~edge
quarantine.append(with_original_values(
    clean[edge].assign(reason="confirmed error at the edge of the history (not extrapolated)"), before_repair))
quarantine.append(with_original_values(
    clean[stale].assign(reason="adjusted close still inconsistent after the repairs"), before_repair))
clean = clean[~edge & ~stale]
checks["after bad-print interpolation (final)"] = consistency_counts(clean)

quarantine = pd.concat(quarantine, ignore_index=True)[KEY + VALUE_COLS + ["reason"]]
step_counts["quarantined rows"] = len(quarantine)
step_counts["retained rows"] = len(clean)
step_counts

adj_close rebuilt: 18374 missing, 0 zero, negative or infinite, 10417 corrupted, 0 after a raw-price repair; 1 suspected spike(s) matched the reference and were kept; 0 unresolved


{'raw rows': 397711,
 'exact duplicates removed': 29460,
 'conflicting-key rows quarantined': 0,
 'all-prices-missing rows dropped': 0,
 'confirmed bad prints interpolated': 3,
 'quarantined rows': 5,
 'retained rows': 368246}

### Recheck after every repair step

The same counts are recomputed after each step. Missing and corrupted adjusted closes are expected until the `adj_close` step; in the final row every count except `rows` must be 0 (asserted again in the validation section).

In [21]:
pd.DataFrame(checks).T

,rows,missing raw price,nonpositive or non-finite raw price,OHLC bounds broken,missing adj_close,"zero, negative or infinite adj_close",adj_close factor above 1 or spike
before repairs,368251,0,5,5,18374,0,10418
after price repairs (Option 1 + forward fill),368246,0,0,0,18374,0,10418
after adj_close rebuild,368246,0,0,0,0,0,2
after bad-print interpolation (final),368246,0,0,0,0,0,0


In [22]:
quarantine[KEY + PRICE_COLS + ["reason"]]

,ticker,ts,open,high,low,close,reason
0,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,invalid or missing price before the ticker's f...
1,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,invalid or missing price before the ticker's f...
2,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,invalid or missing price before the ticker's f...
3,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,invalid or missing price before the ticker's f...
4,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,invalid or missing price before the ticker's f...


The quarantined rows are the five first observations whose `open` is 0 (PG, CVX, JNJ, MRK on 1962-01-02 and MCD on 1966-07-05). An open of 0 is a placeholder, and with no earlier day there is nothing to carry forward; borrowing another ticker's value is not acceptable, so they are set aside with their reason.

### Flags that stay with the data

Repair flags come from the steps above. The two review flags describe the *cleaned* values: `flag_return_outlier` is the 3σ screen rerun on the cleaned prices (the diagnosis on the raw data stays in the Data Quality Report), and `flag_zero_volume` marks days with volume 0.

In [23]:
clean = clean.sort_values(KEY).reset_index(drop=True)
outliers_clean = detect_return_outliers(clean)
outliers_clean = outliers_clean[outliers_clean["issue_type"] == "return_outlier"]
outlier_keys = set(zip(outliers_clean["ticker"], outliers_clean["issue_date"]))
clean["flag_return_outlier"] = [k in outlier_keys for k in zip(clean["ticker"], clean["ts"])]
clean["flag_zero_volume"] = clean["volume"].eq(0)
flag_cols = [c for c in clean.columns if c.startswith("flag_")]
clean[flag_cols].sum().rename("rows flagged")

flag_ohlc_repaired               0
flag_price_filled                0
flag_volume_filled               0
flag_adj_filled              18374
flag_adj_invalid_repaired        0
flag_adj_spike_repaired      10417
flag_adj_recomputed              0
flag_interpolated                3
flag_return_outlier           5176
flag_zero_volume                74
Name: rows flagged, dtype: int64

### Repair behaviour on injected examples

The first table runs the price, volume and adjusted-close repairs on the `DEMO` table from 1.2, with a small reference table whose adjustment factor is 0.99 on every date:
- 2024-01-04 (all raw prices missing) is dropped before any filling;
- 2024-01-05 (high < close) is blanked and takes the previous valid bar (Option 1); its missing volume becomes 0, with a flag;
- 2024-01-10 (close missing) first receives the previous close, which lies above that day's high, so the recheck replaces the row with the previous day's whole bar;
- both repaired rows get `adj_close` = repaired close × 0.99, so the adjustment convention stays the same.

In [24]:
demo_clean, demo_dropped = drop_rows_without_prices(demo)
demo_clean = mark_impossible_prices(demo_clean)
demo_clean, demo_q = fill_prices_within_ticker(demo_clean)
demo_clean = fill_missing_volume(demo_clean)
demo_reference = pd.DataFrame({"ticker": "DEMO", "ts": demo["ts"],
                               "close": [10.1, 10.4, 10.45, 10.5, 10.7, 10.8, 10.3]})
demo_reference["adj_close"] = demo_reference["close"] * 0.99
demo_clean, demo_adj_q = rebuild_adjusted_close(demo_clean, demo_reference)
print("dropped (all prices missing):", demo_dropped["ts"].tolist())
print("recheck after the repairs:", consistency_counts(demo_clean))
demo_clean[["ts"] + PRICE_COLS + ["adj_close", "volume", "flag_ohlc_repaired", "flag_price_filled",
                                  "flag_volume_filled", "flag_adj_recomputed"]]

adj_close rebuilt: 0 missing, 0 zero, negative or infinite, 0 corrupted, 2 after a raw-price repair; 0 suspected spike(s) matched the reference and were kept; 0 unresolved
dropped (all prices missing): ['2024-01-04']
recheck after the repairs: {'rows': 6, 'missing raw price': 0, 'nonpositive or non-finite raw price': 0, 'OHLC bounds broken': 0, 'missing adj_close': 0, 'zero, negative or infinite adj_close': 0, 'adj_close factor above 1 or spike': 0}


,ts,open,high,low,close,adj_close,volume,flag_ohlc_repaired,flag_price_filled,flag_volume_filled,flag_adj_recomputed
0,2024-01-02,10.0,10.5,9.8,10.1,10.000,1000.0,False,False,False,False
1,2024-01-03,10.2,10.6,10.0,10.4,10.300,1200.0,False,False,False,False
2,2024-01-05,10.2,10.6,10.0,10.4,10.296,0.0,True,True,True,True
3,2024-01-08,10.5,10.9,10.2,10.7,10.600,1100.0,False,False,False,False
4,2024-01-09,10.6,11.0,10.4,10.8,10.700,1000.0,False,False,False,False
5,2024-01-10,10.6,11.0,10.4,10.8,10.692,950.0,False,True,False,True


In [25]:
deduped, n_removed = remove_exact_duplicates(demo_keys)
kept, conflicts = split_conflicting_keys(deduped)
print(f"exact duplicates removed: {n_removed}; rows kept: {kept['ts'].tolist()}")
conflicts[KEY + ["close", "reason"]]   # both versions of 2024-03-04 go to quarantine: neither is preferred

exact duplicates removed: 1; rows kept: ['2024-03-01']

,ticker,ts,close,reason
2,DEMO,2024-03-04,20.9,"conflicting values for the same (ticker, ts)"
3,DEMO,2024-03-04,20.6,"conflicting values for the same (ticker, ts)"


The next example isolates invalid adjusted closes on rows whose raw prices are all valid: 0, −2 and −∞ are detected, then rebuilt from the reference factor (0.99 here); the row whose date has no reference factor is quarantined instead.

In [26]:
adj_demo = pd.DataFrame({
    "ticker": "ADJ", "ts": ["2024-06-03", "2024-06-04", "2024-06-05", "2024-06-06", "2024-06-07", "2024-06-10"],
    "open": [50.0, 50.5, 51.0, 51.2, 51.5, 51.7], "high": [50.8, 51.2, 51.6, 51.9, 52.0, 52.3],
    "low": [49.7, 50.1, 50.6, 50.9, 51.1, 51.4], "close": [50.4, 51.0, 51.3, 51.6, 51.8, 52.0],
    "adj_close": [49.896, 0.0, -2.0, -np.inf, 0.0, 51.48],   # 06-04 to 06-07 invalid; every raw price is valid
    "volume": [1000, 1100, 1200, 1300, 1400, 1500],
})
adj_reference = adj_demo[KEY + ["close"]].assign(adj_close=lambda d: d["close"] * 0.99)
adj_reference = adj_reference[adj_reference["ts"] != "2024-06-07"]   # no reference row for 2024-06-07
display(detect_price_problems(adj_demo))
adj_fixed, adj_quarantined = rebuild_adjusted_close(adj_demo, adj_reference)
display(adj_fixed[["ts", "close", "adj_close", "flag_adj_invalid_repaired"]])
adj_quarantined[["ts", "adj_close", "reason"]]

,ticker,issue_type,issue_date,description,severity
0,ADJ,invalid_adj_close,2024-06-04,adj_close = 0.0 (must be finite and positive),high
1,ADJ,invalid_adj_close,2024-06-05,adj_close = -2.0 (must be finite and positive),high
2,ADJ,invalid_adj_close,2024-06-06,adj_close = -inf (must be finite and positive),high
3,ADJ,invalid_adj_close,2024-06-07,adj_close = 0.0 (must be finite and positive),high


adj_close rebuilt: 0 missing, 3 zero, negative or infinite, 0 corrupted, 0 after a raw-price repair; 0 suspected spike(s) matched the reference and were kept; 1 unresolved


,ts,close,adj_close,flag_adj_invalid_repaired
0,2024-06-03,50.4,49.896,False
1,2024-06-04,51.0,50.490,True
2,2024-06-05,51.3,50.787,True
3,2024-06-06,51.6,51.084,True
5,2024-06-10,52.0,51.480,False


,ts,adj_close,reason
4,2024-06-07,0.0,"adj_close missing, invalid, corrupted or stale..."


In [27]:
spike_demo = pd.DataFrame({"ticker": "DEMO", "ts": ["2024-02-01", "2024-02-02", "2024-02-05", "2024-02-06"],
                           "close": [50.0, 75.0, 51.0, 52.0]})
confirmed = pd.Series([False, True, False, False], index=spike_demo.index)   # 75.0 confirmed as an entry error
interpolate_confirmed_errors(spike_demo, confirmed)

,ticker,ts,close,flag_interpolated
0,DEMO,2024-02-01,50.0,False
1,DEMO,2024-02-02,50.5,True
2,DEMO,2024-02-05,51.0,False
3,DEMO,2024-02-06,52.0,False


## Validation

Re-run the detectors on the cleaned table and assert the hard requirements.

In [28]:
report_after = pd.concat([
    detect_duplicates(clean),
    detect_missing_values(clean),
    detect_missing_trading_days(clean, calendar),
    detect_price_problems(clean),
    detect_adj_close_spikes(clean),
    detect_zero_volume(clean),
    detect_return_outliers(clean),
], ignore_index=True)

prices = clean[ALL_PRICE_COLS]
assert prices.notna().all().all(), "missing prices remain"
assert np.isfinite(prices.to_numpy()).all() and (prices > 0).all().all(), "non-finite or nonpositive prices remain"
assert not clean.duplicated(KEY).any(), "duplicate (ticker, ts) keys remain"
assert np.isfinite(clean["volume"]).all() and (clean["volume"] >= 0).all(), "invalid volume"
assert ((clean["low"] <= clean[["open", "close"]].min(axis=1)) & (clean[["open", "close"]].max(axis=1) <= clean["high"])).all()
assert (clean["adj_close"] / clean["close"] <= 1 + 1e-6).all(), "adjustment factor above 1"
assert not unresolved_adjustment(clean, reference).any(), "inconsistent adjusted close remains"
assert clean["ts"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all(), "ts is not stored as ISO YYYY-MM-DD"
assert len(raw) == step_counts["raw rows"], "raw table changed"
assert step_counts["raw rows"] == len(clean) + n_dupes + len(quarantine), "row accounting does not add up"
print("All validation checks pass.")

compare = (summary_before.set_index(["issue_type", "severity"])["issues"].rename("before").to_frame()
           .join(report_after.groupby(["issue_type", "severity"]).size().rename("after"), how="outer")
           .fillna(0).astype(int).sort_values("before", ascending=False))
compare

All validation checks pass.


,,before,after
issue_type,severity,,
duplicate_exact,medium,29460,0
missing_adj_close,high,18374,0
adj_close_spike,high,10418,0
return_outlier,low,5056,5176
ticker_markup,medium,120,0
zero_volume,low,75,74
invalid_open,high,5,0
open_outside_range,high,5,0


In [29]:
remaining = report_after[report_after["issue_type"] == "adj_close_spike"][["ticker", "issue_date"]].rename(columns={"issue_date": "ts"})
remaining.merge(clean, on=KEY).merge(reference[KEY + ["adj_close"]], on=KEY, suffixes=("", "_reference"))[
    KEY + ["close", "adj_close", "adj_close_reference"]]

,ticker,ts,close,adj_close,adj_close_reference


What remains is intentional:
- `return_outlier` and `zero_volume` rows are review flags on values that agree with the supplied data, not errors; they are the rows marked by `flag_return_outlier` and `flag_zero_volume`. The outlier count *rises* after cleaning: the three TRV bad prints (±100% days) inflated TRV's standard deviation, so once they are repaired more ordinary TRV moves cross 3σ. That is the screen working as intended, not new damage.
- An `adj_close_spike` listed above would have an adjusted close equal to the reference value (a genuine step in the adjustment factor, such as a large dividend), since every other one is repaired or quarantined. The table is empty here.

### Before / after statistics

In [30]:
before_stats = dedup_view[VALUE_COLS].describe().T[["count", "mean", "min", "max"]].add_suffix(" (raw, dedup)")
after_stats = clean[VALUE_COLS].describe().T[["count", "mean", "min", "max"]].add_suffix(" (cleaned)")
pd.concat([before_stats, after_stats], axis=1).round(3)

,"count (raw, dedup)","mean (raw, dedup)","min (raw, dedup)","max (raw, dedup)",count (cleaned),mean (cleaned),min (cleaned),max (cleaned)
open,368251.0,4.630900e+01,0.000,6.042600e+02,368246.0,4.630900e+01,0.005,6.042600e+02
high,368251.0,4.677100e+01,0.005,6.086300e+02,368246.0,4.677200e+01,0.005,6.086300e+02
low,368251.0,4.584000e+01,0.005,5.978800e+02,368246.0,4.584000e+01,0.005,5.978800e+02
close,368251.0,4.631700e+01,0.005,6.054000e+02,368246.0,4.631700e+01,0.005,6.054000e+02
adj_close,349877.0,5.171200e+01,0.003,1.044251e+04,368246.0,3.727500e+01,0.003,6.054000e+02
volume,368251.0,3.095142e+07,0.000,9.276606e+09,368246.0,3.095184e+07,0.000,9.276606e+09


## Save

`ts` stays an ISO `YYYY-MM-DD` string so that the report's SQL filters (`ts BETWEEN ? AND ?`) work. The raw table is left unchanged.

In [31]:
columns = ["ts", "ticker"] + VALUE_COLS + flag_cols
to_save = clean[columns].copy()
to_save[flag_cols] = to_save[flag_cols].astype(int)
with sqlite3.connect(db_path) as con:
    to_save.to_sql(cleaned_table, con, if_exists="replace", index=False)
    quarantine.to_sql(quarantine_table, con, if_exists="replace", index=False)
    audit.to_sql(audit_table, con, if_exists="replace", index=False)
    con.execute(f"CREATE INDEX IF NOT EXISTS idx_{cleaned_table}_ticker_ts ON {cleaned_table}(ticker, ts)")
    saved = pd.read_sql_query(f"SELECT COUNT(*) AS n, MIN(ts) AS first, MAX(ts) AS last, COUNT(DISTINCT ticker) AS tickers FROM {cleaned_table}", con)
    raw_now = pd.read_sql_query(f"SELECT * FROM {raw_table}", con)
assert len(raw_now) == raw_row_count and raw_now.equals(raw), "the raw table changed"
saved

,n,first,last,tickers
0,368246,1962-01-02,2024-12-06,30


### Row accounting

Raw rows = retained rows + exact duplicates removed + quarantined rows (asserted in the validation cell).

In [32]:
pd.Series(step_counts)

raw rows                             397711
exact duplicates removed              29460
conflicting-key rows quarantined          0
all-prices-missing rows dropped           0
confirmed bad prints interpolated         3
quarantined rows                          5
retained rows                        368246
dtype: int64